# Retrieval-Augmented Generation (RAG)

This ipython notebook will show how to implement and use Retrieval-Augmented Generation (RAG) for question answering tasks, with a focus on leveraging external knowledge sources to improve the performance of language models.


## Step 1: Install Required Libraries and load pages from the PDF files

In [16]:
from pathlib import Path
from pypdf import PdfReader
import numpy as np
from dotenv import load_dotenv
from google import genai
import json
from google.genai import types
import time
import chromadb
from datetime import datetime
import re




In [2]:
load_dotenv()
client = genai.Client()

In [3]:
def load_pages(folder="data"):
    pages = []
    for pdf in sorted(Path(folder).glob("*.pdf")):
        reader = PdfReader(pdf)
        for i, page in enumerate(reader.pages, start=1):
            text = page.extract_text() or ""
            pages.append({"source": pdf.name, "page": i, "text": text})
    return pages

pages = load_pages()
print(len(pages), "pages")

Ignoring wrong pointing object 9 0 (offset 0)
Ignoring wrong pointing object 26 0 (offset 0)
Ignoring wrong pointing object 102 0 (offset 0)


160 pages


## Step 2: Divide the obtained text into chunks of similar sizes

In [4]:
def chunk_text(text, size=800, overlap=100):
    chunks = []
    start = 0
    n = len(text)
    while start < n:
        end = min(start + size, n)
        if end < n:
            # avoid cutting in the middle of a word
            last_space = text.rfind(" ", start, end)
            if last_space > start + size // 2:
                end = last_space
        piece = text[start:end].strip()
        if piece:
            chunks.append(piece)
        if end >= n:
            break
        start = end - overlap
    return chunks


def build_chunks(pages, size=800, overlap=100, min_len=100):
    chunks = []
    for p in pages:
        for n, piece in enumerate(chunk_text(p["text"], size, overlap), start=1):
            if len(piece) < min_len:
                continue
            chunks.append({
                "id": f'{p["source"]}-p{p["page"]}-c{n}',
                "text": piece,
                "source": p["source"],
                "page": p["page"],
            })
    return chunks


chunks = build_chunks(pages)
print(len(chunks), "chunks")
assert len({c["id"] for c in chunks}) == len(chunks), "duplicate ids"

489 chunks


## Step 3: Sanity Check

Check if the generated chunks picked at random make sense to read, as well to check if the overlap is working.

In [5]:
import random
for c in random.sample(chunks, 5):
    print(c["id"], len(c["text"]))
    print(c["text"], "\n---")

Automated_Dressing_Saturation_Analysis.pdf-p61-c1 798
4.1 Segmentation Results45
ground-truth bounding box prompt. The first variant showed potential, as it achieved a foreground
Dice of 74.65% and a foreground IoU of 70.95% for the exudate, which is just slightly lower
than the transformer architectures. As for the prompted configuration, even though it performed
exceptionally on the exudate task, it will not be directly comparable with the others, since it
relies on a bounding box at inference time, and it is only included to show the upper bound of
that architecture. The hyperparameters and augmentation group selected for each model in these
experiments are reported in Table 4.8.
Table 4.5: Binary dressing segmentation results with data augmentation, for the best architectures.
Best value per metric in bold.
Dice IoU
Model bg fg mDice bg 
---
Automated_Dressing_Saturation_Analysis.pdf-p43-c5 264
face, specially the necessity of an extra input for the bounding box, the reliance
on th

In [6]:
for x, y in zip(chunks, chunks[1:]):
    if x["source"] == y["source"] and x["page"] == y["page"]:
        print(x["id"], y["id"])
        print(repr(x["text"][-120:]))
        print(repr(y["text"][:120]))
        print("overlap found:", y["text"][:60] in x["text"])
        break

Automated_Dressing_Saturation_Analysis.pdf-p4-c1 Automated_Dressing_Saturation_Analysis.pdf-p4-c2
'ão procura\ncontribuir para esta área através da aplicação de técnicas de visão computacional para avaliar a\nsaturação do'
'r para esta área através da aplicação de técnicas de visão computacional para avaliar a\nsaturação do penso.\nFoi conduzid'
overlap found: True


## Step 4: Store the chunks in a JSON file


In [ ]:
with open("chunks.json", "w", encoding="utf-8") as f:
    json.dump(chunks, f, ensure_ascii=False, indent=2)

## Step 5: Generate Embeddings and Store in ChromaDB

In [9]:

EMBEDDING_MODEL = "gemini-embedding-001"

def get_embeddings(texts, task="RETRIEVAL_DOCUMENT") -> np.ndarray:
    response = client.models.embed_content(
        model=EMBEDDING_MODEL,
        contents=texts,
        config=types.EmbedContentConfig(task_type=task),
    )
    assert len(response.embeddings) == len(texts), "embedding count mismatch"
    return np.array([e.values for e in response.embeddings])



In [ ]:
vecs = get_embeddings([c["text"] for c in chunks[:3]])
print(vecs.shape)   # (3, vector_length)

In [7]:
db = chromadb.PersistentClient(path="chroma_db")
col = db.get_or_create_collection(name="docs", metadata={"hnsw:space": "cosine"})
print(col.count())

489


In [27]:
def embed_and_store(chunks, col, batch_size=50, per_minute=90):
    done = set(col.get(ids=[c["id"] for c in chunks])["ids"])
    todo = [c for c in chunks if c["id"] not in done]
    print(f"{len(done)} already stored, {len(todo)} to embed")

    for i in range(0, len(todo), batch_size):
        batch = todo[i:i + batch_size]
        for attempt in range(4):
            try:
                vecs = get_embeddings([c["text"] for c in batch]).tolist()
                break
            except Exception as e:
                wait = 20 * (attempt + 1)
                print(f"failed: {str(e)[:100]} | retrying in {wait}s")
                time.sleep(wait)
        else:
            raise RuntimeError("batch failed after 4 attempts")

        col.upsert(
            ids=[c["id"] for c in batch],
            documents=[c["text"] for c in batch],
            embeddings=vecs,
            metadatas=[{"source": c["source"], "page": c["page"]} for c in batch],
        )
        print(f"stored {min(i + batch_size, len(todo))}/{len(todo)}")
        time.sleep(60 * len(batch) / per_minute)   # about 33s for 50 texts



In [ ]:
embed_and_store(chunks, col)
print(col.count(), "items in collection")

# Step 6: Query Testing


In [10]:
def retrieve(question, k=4):
    qv = get_embeddings([question], task="RETRIEVAL_QUERY")[0].tolist()
    res = col.query(query_embeddings=[qv], n_results=k)
    return list(zip(res["documents"][0], res["metadatas"][0], res["distances"][0]))

for doc, meta, dist in retrieve("O que é uma ferida crónica?"):
    print(round(dist, 3), meta)
    print(doc[:200], "\n---")


#Questions asked
## What kind of political system does Portugal have?
## What is a chronic wound?
## Who is Ryze?
## How do I bake a cake?
## O que é uma ferida crónica?


0.265 {'source': 'Automated_Dressing_Saturation_Analysis.pdf', 'page': 17}
Chapter 1
Introduction
1.1 Context
A chronic wound is defined as a wound that does not follow a normal healing process, nor-
mally taking more than 4 to 12 weeks without reaching full anatomic and fun 
---
0.282 {'page': 17, 'source': 'Automated_Dressing_Saturation_Analysis.pdf'}
ility
limitations and the risk of recurrent infections. Multiple observational studies have reported that
the presence of chronic wounds of various types is estimated at 2.21 per 1000 people [38], al- 
---
0.296 {'page': 4, 'source': 'Automated_Dressing_Saturation_Analysis.pdf'}
Resumo
As feridas crónicas aplicam um peso significativo aos sistemas de saúde e à qualidade de vida dos
doentes. Durante o processo de cicatrização das feridas crónicas, o excesso de exsudado faz com 
---
0.318 {'source': 'Automated_Dressing_Saturation_Analysis.pdf', 'page': 87}
cent Methods for
Non-Contact Chronic Wound Analysis”. In:Applied Sciences10.21 (202

In [11]:
EMBEDDING_MODEL = "gemini-embedding-001"
GENERATION_MODEL = "gemini-3.5-flash-lite"
LOG_FILE = "retrieval_logs.jsonl"

# 1. Function retrieve(question, k=4) that generates the embedding, prints and returns the chunks with distances.
def retrieve(question: str, k: int = 4):
    qv = get_embeddings([question], task="RETRIEVAL_QUERY")[0].tolist()
    res = col.query(
        query_embeddings=[qv],
        n_results=k,
        include=["documents", "metadatas", "distances"]
    )
    
    chunks_retrieved = []
    print(f"\n=== Chunks Retrieved for: '{question}' ===")
    for cid, doc, meta, dist in zip(
        res["ids"][0], res["documents"][0], res["metadatas"][0], res["distances"][0]
    ):
        item = {
            "id": cid,
            "text": doc,
            "source": meta["source"],
            "page": meta["page"],
            "distance": round(float(dist), 4),
        }
        chunks_retrieved.append(item)
        print(f"[{item['distance']:.3f}] {item['source']} (page. {item['page']}) | ID: {item['id']}")
        print(f"{item['text'][:200]}...\n---")
        
    return chunks_retrieved

# 2. Auxiliar Function to register the chunks retrieved for each call
def log_call(question: str, chunks: list, answer: str):
    record = {
        "timestamp": datetime.now().isoformat(),
        "question": question,
        "retrieved_chunks": chunks,
        "answer": answer,
    }
    with open(LOG_FILE, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

# 3. Generation function with strict prompt, low temperature and returns the sources as well
def generate_answer(question: str, k: int = 4):
    chunks = retrieve(question, k=k)
    
    # Format the context with the source
    blocks_context = []
    for c in chunks:
        blocks_context.append(
            f"[File: {c['source']}, Page: {c['page']}]\n{c['text']}"
        )
    context_str = "\n\n---\n\n".join(blocks_context)
    
    prompt_system = (
        "You're a rigurous assistant of questions and answers.\n"
        "Mandatory Rules:\n"
        "1. Only answer based on the context given below.\n"
        "2. If the answer isn't explicit in the context, answer with I don't know."
        "(ex: 'I can't answer based on the given documents.'). Do not create new information nor use external knowledge.\n"
        "3. Always cite the file and the page from where you got the information in the answer's body."
        "(format: [File: name.pdf, Page: X])."
    )
    
    prompt_user = f"CONTEXTO:\n{context_str}\n\nQUESTION: {question}"
    
    response = client.models.generate_content(
        model=GENERATION_MODEL,
        contents=prompt_user,
        config=types.GenerateContentConfig(
            system_instruction=prompt_system,
            temperature=0.1,
        ),
    )
    
    text_answer = response.text.strip()
    
    # Extract unique line of consulted sources
    sources = sorted({f"{c['source']} (pág. {c['page']})" for c in chunks})
    final_answer = f"{text_answer}\n\nFontes recuperadas:\n" + "\n".join(f"- {f}" for f in sources)
    
    # Register the chunks and the answers for debugging
    log_call(question, chunks, text_answer)
    
    return {
        "answer": text_answer,
        "sources": sources,
        "chunks": chunks,
        "formatted_output": final_answer,
    }

#Test
result = generate_answer("What ingredients and oven temperature are needed to bake a traditional chocolate cake?", k=4)
print("\n=== Final Answer ===")
print(result["formatted_output"])


=== Chunks Retrieved for: 'What ingredients and oven temperature are needed to bake a traditional chocolate cake?' ===
[0.447] Automated_Dressing_Saturation_Analysis.pdf (page. 10) | ID: Automated_Dressing_Saturation_Analysis.pdf-p10-c1
Contents
1 Introduction 1
1.1 Context . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 1
1.2 Motivation . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . ....
---
[0.459] Automated_Dressing_Saturation_Analysis.pdf (page. 7) | ID: Automated_Dressing_Saturation_Analysis.pdf-p7-c1
Acknowledgements
Aos meus orientadores, prof. Ricardo Cruz e Maria Vasconcelos, e também ao Gil Fernandes,
gostaria de deixar uma palavra de agradecimento pelo apoio e orientação durante este projeto ...
---
[0.460] Automated_Dressing_Saturation_Analysis.pdf (page. 11) | ID: Automated_Dressing_Saturation_Analysis.pdf-p11-c4
. . . . . . . . . 63
4.4.3 Comparison with the Literature . . . . . . . . . . . . . . . . . . . . . . 64
4.4.4 L

# Step 7: Improving the RAG

Given that the Query 3 is failing to succeed, this step has the purpose of trying to understand what's going on and what's going wrong.

In [12]:
SRC = "Automated_Dressing_Saturation_Analysis.pdf"
def find(substr):
    res = col.get(
        where={"source": SRC},
        where_document={"$contains": substr},
        include=["documents", "metadatas"],
    )
    return {i: (d, m) for i, d, m in zip(res["ids"], res["documents"], res["metadatas"])}

# dataset size: chunks with 459 AND a marker that makes them actually state the dataset composition
size_markers = ["281 real", "In total it contains", "out of 459", "154 positives"]
gold_size = {i for i, (d, m) in find("459").items() if any(k in d for k in size_markers)}

# best pipeline: 92.71 together with the recall 93.88 (abstract, results p.50, Table 4.19)
gold_pipe = {i for i, (d, m) in find("92.71").items() if "93.88" in d}

for name, group in [("DATASET SIZE", gold_size), ("BEST PIPELINE", gold_pipe)]:
    print(f"\n== {name} ({len(group)} chunks) ==")
    allhits = {**find("459"), **find("92.71")}
    for i in sorted(group):
        d, m = allhits[i]
        print(i, "| page", m["page"], "|", d[:90].replace("\n", " "))


== DATASET SIZE (5 chunks) ==
Automated_Dressing_Saturation_Analysis.pdf-p39-c1 | page 39 | 3.3 Dataset 23 3.3.1 Dataset Description Given the scarcity of available wound dressing da
Automated_Dressing_Saturation_Analysis.pdf-p39-c4 | page 39 | of exudate. Regarding the classification of the wound dressings, the dataset has a class i
Automated_Dressing_Saturation_Analysis.pdf-p81-c2 | page 81 | e. Nevertheless, the contribution of this thesis remains to be the establishment of the fi
Automated_Dressing_Saturation_Analysis.pdf-p82-c1 | page 82 | Chapter 5 Conclusions and Future Work This dissertation tackled the automated evaluation o
Automated_Dressing_Saturation_Analysis.pdf-p82-c2 | page 82 | specialised dataset of 459 images, with 281 real clinical images of chronic wound dressing

== BEST PIPELINE (5 chunks) ==
Automated_Dressing_Saturation_Analysis.pdf-p4-c4 | page 4 | é a mais complexa. A classificação apenas com RGB, com base no modelo ConvNeXt-Small, al- 
Automated_Dressing_Sa

### Gold chunks (ground truth for the diagnosis)

A *gold chunk* is a chunk that really contains the answer. I found them with an exact text search (`col.get(where_document={"$contains": ...})`), not with embeddings, so they do not depend on the retrieval being tested.

- **Dataset size:** chunks containing `459` **and** a marker that states the dataset composition (`281 real`, `In total it contains`, `out of 459`, `154 positives`). 5 chunks: Section 3.3.1 and the class-imbalance paragraph (page 39), the Limitations paragraph (page 81) and the two Conclusions chunks (page 82).
- **Best pipeline:** chunks containing both `92.71` and `93.88`. 5 chunks: the Portuguese and English abstracts (pages 4 and 5), the Results prose and Table 4.19 (page 66) and Table 4.32 (page 79).
- **Excluded on purpose:** the reference-list hit (`pp. 459–479`, a false positive for `459`) and the Table 3.1 row `Total 459`, which has no context around it.

**Page numbers:** the `page` stored in Chroma is the PDF page index. In the body of the dissertation the printed page is the PDF page minus 16 (PDF page 39 is printed page 23, PDF page 66 is printed page 50). In the front matter the offset is different (roman numerals), so it is not constant. The eval set has to use PDF pages or chunk ids, not the printed numbers.

In [13]:
_cache = {}

def rank_all(question):
    """All chunks ranked by distance for this question (embedding cached)."""
    if question not in _cache:
        qv = get_embeddings([question], task="RETRIEVAL_QUERY")[0].tolist()
        res = col.query(
            query_embeddings=[qv],
            n_results=col.count(),
            include=["distances", "metadatas"],
        )
        _cache[question] = list(zip(res["ids"][0], res["distances"][0], res["metadatas"][0]))
    return _cache[question]

def rank_report(label, question, gold, show_top=5):
    ranked = rank_all(question)
    ids = [r[0] for r in ranked]
    pos = sorted(ids.index(g) + 1 for g in gold if g in ids)   # 1-based ranks of gold chunks
    best = pos[0] if pos else None
    best_dist = ranked[best - 1][1] if best else None
    print(f"\n### {label}")
    print(f"gold ranks: {pos}")
    print(f"best rank: {best} (distance {best_dist:.3f})" if best else "best rank: none")
    for k in (4, 8, 10, 20):
        n = sum(1 for p in pos if p <= k)
        print(f"  top-{k}: {n}/{len(gold)} gold chunks")
    print("  distance at rank 1/4/10/20:", [round(ranked[r - 1][1], 3) for r in (1, 4, 10, 20)])
    print(f"  top {show_top}:")
    for r, (cid, dist, meta) in enumerate(ranked[:show_top], start=1):
        tag = " <-- GOLD" if cid in gold else ""
        print(f"   {r}. {dist:.3f} {cid} (page {meta['page']}){tag}")

In [14]:
Q_FULL = ("How many images are in the dissertation's dataset, and which pipeline "
          "achieved the highest overall F-beta score on the fold 0 test set?")
Q_A = "How many images are in the dissertation's dataset?"
Q_B = "Which pipeline achieved the highest overall F-beta score on the fold 0 test set?"

rank_report("FULL question vs dataset-size gold", Q_FULL, gold_size)
rank_report("FULL question vs best-pipeline gold", Q_FULL, gold_pipe)
rank_report("Sub-question A vs dataset-size gold", Q_A, gold_size)
rank_report("Sub-question B vs best-pipeline gold", Q_B, gold_pipe)



### FULL question vs dataset-size gold
gold ranks: [30, 34, 67, 74, 137]
best rank: 30 (distance 0.266)
  top-4: 0/5 gold chunks
  top-8: 0/5 gold chunks
  top-10: 0/5 gold chunks
  top-20: 0/5 gold chunks
  distance at rank 1/4/10/20: [0.231, 0.25, 0.255, 0.263]
  top 5:
   1. 0.231 Automated_Dressing_Saturation_Analysis.pdf-p41-c3 (page 41)
   2. 0.249 Automated_Dressing_Saturation_Analysis.pdf-p5-c4 (page 5)
   3. 0.250 Automated_Dressing_Saturation_Analysis.pdf-p78-c2 (page 78)
   4. 0.250 Automated_Dressing_Saturation_Analysis.pdf-p70-c2 (page 70)
   5. 0.251 Automated_Dressing_Saturation_Analysis.pdf-p80-c1 (page 80)

### FULL question vs best-pipeline gold
gold ranks: [2, 9, 13, 139, 152]
best rank: 2 (distance 0.249)
  top-4: 1/5 gold chunks
  top-8: 1/5 gold chunks
  top-10: 2/5 gold chunks
  top-20: 3/5 gold chunks
  distance at rank 1/4/10/20: [0.231, 0.25, 0.255, 0.263]
  top 5:
   1. 0.231 Automated_Dressing_Saturation_Analysis.pdf-p41-c3 (page 41)
   2. 0.249 Automated_D

### Q3 diagnosis: results

Gold ranks are positions among all 489 chunks (1 = closest). The lists are sorted, so this output shows *how many* gold chunks sit where, but not *which* chunk is which rank.

| Query | Gold group | Gold ranks | Gold in top 4 / 10 / 20 |
|---|---|---|---|
| Full question (`Q_FULL`) | dataset size | 30, 34, 67, 74, 137 | 0 / 0 / 0 |
| Full question (`Q_FULL`) | best pipeline | 2, 9, 13, 139, 152 | 1 / 2 / 3 |
| Sub-question A (dataset size) | dataset size | 1, 9, 12, 13, 25 | 1 / 2 / 4 |
| Sub-question B (best pipeline) | best pipeline | 19, 20, 75, 117, 140 | 0 / 0 / 2 |

**What the numbers say**

1. **The dataset-size half is a retrieval failure.** For the full question the best dataset-size chunk is at rank 30. A larger `k` cannot realistically fix this: `k=20` still finds 0 of 5, and `k=30` would put about 6% of the whole collection into the prompt.
2. **The F-beta half was not a retrieval failure.** An abstract chunk with 92.71% and 93.88% was already at rank 2, inside the top 4 of the original run. The original answer was still "I can't answer", so that refusal came from the prompt rule ("if the answer isn't explicit, I don't know"), which refused the whole question because one half was missing. Q3 therefore had two causes at once: a retrieval problem for one half and a prompt problem for the other.
3. **Blending hurts the dataset-size half.** The best dataset-size chunk moves from rank 30 (full question) to rank 1 (sub-question A). The likely reason is that one embedding for two topics sits between both targets, but I did not test that directly.
4. **Sub-question B alone is worse than the full question** for the best-pipeline gold (best rank 19 against 2). I did not establish why. Two hypotheses to test: many chunks discuss F-beta scores of the other approaches, so the topic is crowded; or the full question has extra words (`dataset`, `dissertation`) that pull toward the abstract. This matters for decomposition: if only the sub-queries are used with a small `k` per query, the F-beta half can get lost.
5. **The distance curve is flat.** Rank 1 to rank 20 spans only about 0.03 for the full question (0.231 to 0.263) and for B (0.244 to 0.276), and about 0.06 for A (0.226 to 0.283). Ranks are volatile with such small gaps, and a distance threshold cannot separate good from bad chunks here.
6. **Front matter shows up, but was not the main cause for Q3.** In the later `k=8` run a List of Tables chunk (page 14) is retrieved at position 6, which supports cleaning the corpus, but the abstract chunk was still ranked above it.

# Step 8: Adding query decomposition and partial answering to the RAG

**Plan for Step 8.** Q3 has two stacked causes (Step 7), so it needs two kinds of fix:

1. **Prompt (generation):** rule 3 now allows partial answers: answer what the context supports and state which part cannot be answered. "I don't know" stays for questions where nothing relevant appears.
2. **Retrieval:** `k=8` as a first attempt, then `decompose_query` (Gemini splits the question into single-topic sub-queries, with structured output) and `retrieve_multi` (retrieve per sub-query and merge without duplicates).

Because of finding 4 in Step 7, the merge should probably also keep the results of the **original full question**, not only of the sub-queries.

In [15]:
from pydantic import BaseModel
import json

class DecomposedQuery(BaseModel):
    sub_queries: list[str]

def decompose_query(question: str) -> list[str]:
    system_instruction = (
        "You are a search query optimizer. Break the user's complex, multi-part question "
        "into independent, single-topic questions that are optimized for semantic vector search. "
        "If the question is already simple and single-topic, return a list containing just that question."
    )
    
    response = client.models.generate_content(
        model=GENERATION_MODEL,
        contents=question,
        config=types.GenerateContentConfig(
            system_instruction=system_instruction,
            temperature=0.0,
            response_mime_type="application/json",
            response_schema=DecomposedQuery,
        ),
    )
    
    return json.loads(response.text)["sub_queries"]

# Example usage:
# sub_queries = decompose_query("How many images are in the dataset and which pipeline won on fold 0?")
# Returns: ["How many images are in the dataset?", "Which pipeline won on fold 0?"]

**Status of the changes in the next cell:**

- Partial-answer prompt (rule 3): active in `generate_answer`.
- `retrieve_multi` and `decompose_query`.

In [53]:
EMBEDDING_MODEL = "gemini-embedding-001"
GENERATION_MODEL = "gemini-3.5-flash-lite"
LOG_FILE = "retrieval_logs.jsonl"

# 1. Function retrieve(question, k=4) that generates the embedding, prints and returns the chunks with distances.
def retrieve(question: str, k: int = 4):
    qv = get_embeddings([question], task="RETRIEVAL_QUERY")[0].tolist()
    res = col.query(
        query_embeddings=[qv],
        n_results=k,
        include=["documents", "metadatas", "distances"]
    )
    
    chunks_retrieved = []
    print(f"\n=== Chunks Retrieved for: '{question}' ===")
    for cid, doc, meta, dist in zip(
        res["ids"][0], res["documents"][0], res["metadatas"][0], res["distances"][0]
    ):
        item = {
            "id": cid,
            "text": doc,
            "source": meta["source"],
            "page": meta["page"],
            "distance": round(float(dist), 4),
        }
        chunks_retrieved.append(item)
        print(f"[{item['distance']:.3f}] {item['source']} (page. {item['page']}) | ID: {item['id']}")
        print(f"{item['text'][:200]}...\n---")
        
    return chunks_retrieved

# 2. Auxiliar Function to register the chunks retrieved for each call
def log_call(question: str, chunks: list, answer: str):
    record = {
        "timestamp": datetime.now().isoformat(),
        "question": question,
        "retrieved_chunks": chunks,
        "answer": answer,
    }
    with open(LOG_FILE, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

# 3. NEW retrieval multi-query assistant
def retrieve_multi(question: str, k_per_query: int = 3) -> list[dict]:
    sub_queries = decompose_query(question)
    print(f"\n[Query Decomposition] Original: '{question}'")
    print(f"[Query Decomposition] Sub-queries: {sub_queries}")
    
    all_chunks = {}
    
    for sub_q in sub_queries:
        # Calls your existing retrieve() function
        chunks = retrieve(sub_q, k=k_per_query) 
        for c in chunks:
            # Deduplicate by chunk ID so the LLM doesn't read the same text twice
            if c['id'] not in all_chunks:
                all_chunks[c['id']] = c
                
    final_chunks = list(all_chunks.values())
    print(f"\n[Query Decomposition] Total unique chunks retrieved: {len(final_chunks)}\n")
    
    return final_chunks

# 4. Generation function with strict prompt, low temperature and returns the sources as well
def generate_answer(question: str, k: int = 4):
    chunks = retrieve_multi(question, k_per_query=k)
    
    # Format the context with the source
    blocks_context = []
    for c in chunks:
        blocks_context.append(
            f"[File: {c['source']}, Page: {c['page']}]\n{c['text']}"
        )
    context_str = "\n\n---\n\n".join(blocks_context)
    
    prompt_system = (
        "You're a rigorous assistant of questions and answers.\n"
        "Mandatory Rules:\n"
        "1. Only answer based on the context given below.\n"
        "2. If the answer isn't explicit in the context, answer with I don't know "
        "(ex: 'I can't answer based on the given documents.'). Do not create new information nor use external knowledge.\n"
        "3. You are allowed to give partial answers. If the context only supports part of a multi-part question, answer what you can and explicitly state which part of the question you cannot answer based on the documents.\n"
        "4. Always cite the file and the page from where you got the information in the answer's body "
        "(format: [File: name.pdf, Page: X])."
    )
    
    prompt_user = f"CONTEXTO:\n{context_str}\n\nQUESTION: {question}"
    
    response = client.models.generate_content(
        model=GENERATION_MODEL,
        contents=prompt_user,
        config=types.GenerateContentConfig(
            system_instruction=prompt_system,
            temperature=0.1,
        ),
    )
    
    text_answer = response.text.strip()
    
    # Extract unique line of consulted sources
    sources = sorted({f"{c['source']} (pág. {c['page']})" for c in chunks})
    final_answer = f"{text_answer}\n\nFontes recuperadas:\n" + "\n".join(f"- {f}" for f in sources)
    
    # Register the chunks and the answers for debugging
    log_call(question, chunks, text_answer)
    
    return {
        "answer": text_answer,
        "sources": sources,
        "chunks": chunks,
        "formatted_output": final_answer,
    }

#Test


In [ ]:
result = generate_answer("How many images are in the dissertation's dataset, and which pipeline achieved the highest overall F-beta score on the fold 0 test set?", k=8)
print("\n=== Final Answer ===")
print(result["formatted_output"])

# Step 9: Cleaning the Corpus

In [23]:

def scan(source):
    for p in pages:
        if p["source"] != source:
            continue
        t = p["text"]
        lines = [l for l in t.split("\n") if l.strip()]
        endnum = sum(bool(re.search(r"\d\s*$", l)) for l in lines) / max(len(lines), 1)
        dots = t.count(". . .")
        refs = len(re.findall(r"^\[\d+\]", t, re.M))
        print(f'{p["page"]:>3} len={len(t):>5} dots={dots:>2} refs={refs:>2} '
              f'endnum={endnum:.2f} | {t[:50]!r}')

print({s: sum(p["source"] == s for p in pages) for s in sorted({p["source"] for p in pages})})
#scan("Automated_Dressing_Saturation_Analysis.pdf")
#scan("league-of-legends-beginners-guide_compress.pdf")
#scan("python.pdf")
scan("Revolucao_cravos.pdf")

{'Automated_Dressing_Saturation_Analysis.pdf': 88, 'Revolucao_cravos.pdf': 22, 'league-of-legends-beginners-guide_compress.pdf': 36, 'python.pdf': 14}
  1 len=  501 dots= 0 refs= 0 endnum=0.20 | ' \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \nCenter f'
  2 len= 2827 dots= 0 refs= 0 endnum=0.06 | 'Portuguese Exceptionalism and the Return to Europe'
  3 len= 2475 dots= 0 refs= 0 endnum=0.00 | 'space, trapped in the historical and philosophical'
  4 len= 2547 dots= 0 refs= 0 endnum=0.00 | 'century, which he refers to as ‘waves.’  Huntingto'
  5 len= 2310 dots= 0 refs= 0 endnum=0.03 | 'of elections (Manuel 1995: 21–23). For instance, v'
  6 len= 2607 dots= 0 refs= 0 endnum=0.00 | 'ascribe a variety of sometimes conflicting meaning'
  7 len= 2417 dots= 0 refs= 0 endnum=0.03 | 'political rights of its citizens (see Schumpeter 1'
  8 len= 2413 dots= 0 refs= 0 endnum=0.00 | 'outcome; the victorious faction was able to shape '
  9 len= 2722 dots= 0 refs= 0 endnum=0.09 | 'gaining

From the observation done above, the results will be recorded as inclusive page ranges per PDF, which will filter out the contents and lists of tables/figures, references, index, acronyms and short/blank pages.

In [24]:
SKIP = {
    "Automated_Dressing_Saturation_Analysis.pdf" : [(10,16),(85,88)],
    "league-of-legends-beginners-guide_compress.pdf" : [(2,5),(34,36)],
    "python.pdf" : [],
    "Revolucao_cravos.pdf" : [(20,22)]
}

With the SKIP dict defined above, we write the cleaning function.

In [25]:
def in_skip(source, page):
    return any(a <= page <= b for a, b in SKIP.get(source, []))

# typo guard: every key must be a real file name
sources = {p["source"] for p in pages}
assert set(SKIP) <= sources, f"unknown keys: {set(SKIP) - sources}"

# gold pages from Step 7 must survive
S = "Automated_Dressing_Saturation_Analysis.pdf"
assert not any(in_skip(S, pg) for pg in (4, 5, 39, 66, 79, 81, 82)), "a gold page is being skipped"

pages_kept = [p for p in pages if not in_skip(p["source"], p["page"])]   # page numbers untouched
print(len(pages), "->", len(pages_kept), "pages")

# boundary check: page before the range, first and last skipped page, page after
text_of = {(p["source"], p["page"]): p["text"] for p in pages}
for src, ranges in SKIP.items():
    for a, b in ranges:
        print(f"\n== {src[:28]}  {a}-{b} ==")
        for pg in (a - 1, a, b, b + 1):
            t = text_of.get((src, pg), "")
            print(f"  p{pg:>3} {'SKIP' if a <= pg <= b else 'keep'} | {t[:70]!r}")

160 -> 139 pages

== Automated_Dressing_Saturatio  10-16 ==
  p  9 keep | '“To dare is to lose one’s footing momentarily. Not to dare is to lose '
  p 10 SKIP | 'Contents\n1 Introduction 1\n1.1 Context . . . . . . . . . . . . . . . . '
  p 16 SKIP | 'LIST OF ACRONYMSxiii\nMPAMean Pixel Accuracy\nNPIAPNational Pressure Inj'
  p 17 keep | 'Chapter 1\nIntroduction\n1.1 Context\nA chronic wound is defined as a wou'

== Automated_Dressing_Saturatio  85-88 ==
  p 84 keep | 'Conclusions and Future Work68\nclassifier, which was tuned on the test '
  p 85 SKIP | 'References\n[1] Takuya Akiba et al. “Optuna: A Next-Generation Hyperpar'
  p 88 SKIP | 'REFERENCES72\n[50] B. Rostami et al. “Multiclass wound image classifica'
  p 89 keep | ''

== league-of-legends-beginners-  2-5 ==
  p  1 keep | 'League of LegendsLeague of Legends\nBeginner’s Guide\nJosh Gregory\nUNOFF'
  p  2 SKIP | ''
  p  5 SKIP | 'Chapter 1  \nA Legendary Game  4\nChapter 2   \nStay in Your Lane  10\nCha'
  p  6 keep | 'CHAPTER

Next, we have the small issues found when PDF's were converted to corpus, such as soft hyphens and newlines. Other issues like missing spaces in italic texts are hard to fix with regex, and will be stated as a limitation.

In [26]:
LOWER = "a-záàâãçéêíóôõúü"

def clean_text(t: str) -> str:
    t = t.replace("\u00ad", "")                                  # soft hyphens
    t = re.sub(rf"([{LOWER}])-\n([{LOWER}])", r"\1\2", t)        # "nor-\nmally" -> "normally"
    t = re.sub(r"(?<!\n)\n(?!\n)", " ", t)                       # single newline -> space
    t = re.sub(r"[ \t]+", " ", t)                                # collapse spaces
    return t.strip()

pages_clean = [{**p, "text": clean_text(p["text"])} for p in pages_kept]
chunks_clean = build_chunks(pages_clean)

HY = re.compile(rf"[{LOWER}]-\n[{LOWER}]")
print("hyphen breaks before:", sum(len(HY.findall(p["text"])) for p in pages_kept))
print("hyphen breaks after :", sum(len(HY.findall(p["text"])) for p in pages_clean))
print("newlines before/after:", sum(p["text"].count("\n") for p in pages_kept),
      "/", sum(p["text"].count("\n") for p in pages_clean))
print("chunks:", len(chunks), "->", len(chunks_clean))
print("gold text still present:", any("281 real" in c["text"] for c in chunks_clean),
      any("92.71" in c["text"] and "93.88" in c["text"] for c in chunks_clean))
assert len({c["id"] for c in chunks_clean}) == len(chunks_clean)

import random
for c in random.sample(chunks_clean, 3):
    print("\n", c["id"], len(c["text"]), "\n", c["text"][:400])

hyphen breaks before: 235
hyphen breaks after : 0
newlines before/after: 3815 / 0
chunks: 489 -> 426
gold text still present: True True

 Automated_Dressing_Saturation_Analysis.pdf-p33-c1 795 
 2.5 Classification Approaches17 CoAtNet [14] is a hybrid architecture that vertically merges Convolutional Neural Networks (CNNs) and Attention based Vision Transformers (ViTs). This design aims to merge the depthwise convolution and the self-attention through a relative attention mechanism, which improves the generalisation, capacity and efficiency. On top of that, it gives this model the inducti

 league-of-legends-beginners-guide_compress.pdf-p14-c1 646 
 Each team’s Nexus is protected by towers called turrets. These towers are located in specific locations on the map. They will attack any enemies who get too close. In addition, each team’s Nexus will spawn a series of computer-controlled creatures called minions. Minions will move toward the opposing team’s Nexus, attacking enemy players, mi

Now, we embed this filtered contents into a new collection.

In [28]:
col_base = col                                   # baseline stays safe under this name
col_clean = db.get_or_create_collection(name="docs_clean", metadata={"hnsw:space": "cosine"})
print("baseline:", col_base.count(), "| clean (before):", col_clean.count())

with open("chunks_clean.json", "w", encoding="utf-8") as f:
    json.dump(chunks_clean, f, ensure_ascii=False, indent=2)

embed_and_store(chunks_clean, col_clean)
print("clean (after):", col_clean.count())

baseline: 489 | clean (before): 0
0 already stored, 426 to embed
stored 50/426
stored 100/426
stored 150/426
stored 200/426
stored 250/426
stored 300/426
stored 350/426
stored 400/426
stored 426/426
clean (after): 426


In [68]:
from rag_eval import run_eval

col = col_base;  print(col.name)       # must print: docs
run_eval(lambda q: generate_answer(q, k=4), label="decomp_k4_docs")

time.sleep(70)

col = col_clean; print(col.name)       # must print: docs_clean
run_eval(lambda q: generate_answer(q, k=4), label="decomp_k4_docs_clean")

col = col_base                         # leave the baseline bound

docs
PASS  Q1  chunks=5   16.7s  
PASS  Q2  chunks=4   8.9s  
FAIL  Q3  chunks=8   15.5s  
PASS  Q4  chunks=6   8.1s  
PASS  Q5  chunks=4   7.1s  
PASS  Q6  chunks=6   7.2s  
FAIL  Q7  chunks=0   2.9s  ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
PASS  Q8  chunks=4   4.8s  
FAIL  Q9  chunks=0   3.8s  ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
PASS  Q10 chunks=6   3.8s  
PASS  Q11 chunks=6   2.8s  
PASS  Q12 chunks=5   2.6s  
PASS  Q13 chunks=5   3.5s  
PASS  Q14 chunks=7   2.7s  
PASS  Q15 chunks=7   2.3s  

| run | retrieval hit | cited hit | negatives refused | avg chunks | errors |
|---|---|---|---|---|---|
| decomp_k4_docs | 9/12 | 9/12 | 3/3 | 4.9 | 2 |
docs_